In [16]:
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
import torch.optim as optim

In [9]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

In [10]:
cwd = Path.cwd()
project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done")

In [11]:
from Scripts.utils import load_mnist_dataset

In [12]:
data_path = project_root / "data"
train_dataloader, test_dataloader = load_mnist_dataset(
    data_path=data_path,
    batch_size=128
)

In [13]:
next(iter(train_dataloader))

[tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         ...,
 
 
         [[[-1., -1., -1.,  ..., -

### Dataset is ready, Let's build the generator

In [14]:
class Generator(nn.Module):
    def __init__(self, noise_size: int):
        super().__init__()

        self.leaky_relu = nn.LeakyReLU()
        self.layer1 = nn.ConvTranspose2d(in_channels=noise_size, out_channels=256, stride=1, padding=0, kernel_size=7)
        self.batch_norm_1 = nn.BatchNorm2d(256)
        self.layer2 = nn.ConvTranspose2d(in_channels=256, out_channels=128, stride=2, padding=1, kernel_size=4)
        self.batch_norm_2 = nn.BatchNorm2d(128)
        self.layer3 = nn.ConvTranspose2d(in_channels=128, out_channels=1, stride=2, padding=1, kernel_size=4)
        self.tanh = nn.Tanh()

    def forward(self, X):
        X = self.layer1(X)          # (128, 256, 7, 7)
        X = self.batch_norm_1(X)    # (128, 256, 7, 7)
        X = self.leaky_relu(X)      # (128, 256, 7, 7)
        X = self.layer2(X)          # (128, 128, 14, 14)
        X = self.batch_norm_2(X)    # (128, 128, 14, 14)
        X = self.leaky_relu(X)      # (128, 128, 14, 14)
        X = self.layer3(X)          # (128, 1, 28, 28)
        X = self.tanh(X)            # (128, 1, 28, 28)

        return X

In [15]:
class Critic(nn.Module):
    def __init__(self):
        super().__init__()

        self.leaky_relu = nn.LeakyReLU(negative_slope=0.2)
        self.layer1 = nn.Conv2d(in_channels=1, out_channels=128, stride=2, padding=1, kernel_size=4)
        self.layer2 = nn.Conv2d(in_channels=128, out_channels=256, stride=2, padding=1, kernel_size=4)
        self.layer3 = nn.Conv2d(in_channels=256, out_channels=1, stride=1, padding=0, kernel_size=7)

    def forward(self, X):
        X = self.layer1(X)      # (128, 128, 14, 14)
        X = self.leaky_relu(X)  # (128, 128, 14, 14)
        X = self.layer2(X)      # (128, 256, 7, 7)
        X = self.leaky_relu(X)  # (128, 256, 7, 7)
        X = self.layer3(X)      # (128, 1, 1, 1)

        return X.view(-1, 1)

### Generator and Critic are ready, Let's Initialize

In [18]:
# Hyperparameters
LEARNING_RATE = 5e-5
NOISE_DIM = 100
BATCH_SIZE = 128
CLIP_VALUE = 0.01
CRITIC_ITERATIONS = 5

# INITIALIZING THE GENERATOR AND THE DISCRIMINATOR
generator = Generator(noise_size=NOISE_DIM).to(device=device)
critic = Critic().to(device=device)

# Initializing the optimizers
opt_gen = optim.RMSprop(params=generator.parameters(), lr=LEARNING_RATE)
opt_crit = optim.RMSprop(params=critic.parameters(), lr=LEARNING_RATE)

In [ ]:
def train(
        device,
        noise_dim: int,
        batch_size: int,
        clip_value: int,
        critic_iterations: int,
        generator: Generator,
        critic: Critic,
        opt_gen: optim.RMSprop,
        opt_crit: optim.RMSprop,
        dataloader,
        epochs: int = 3,
):
    for epoch in range(epochs):
        for batch, _ in dataloader:
            batch = batch.to(device)
            for _ in range(critic_iterations):
                noise = torch.randn(size=[batch_size, noise_dim, 1, 1], device=device)
                fake_images = generator.forward(noise)
                critic_real = critic.forward(batch)
                critic_fake = critic.forward(fake_images)
                critic_loss = -(torch.mean(critic_real) - torch.mean(critic_fake))

                # backward
                opt_crit.zero_grad()
                critic_loss.backward()
                opt_crit.step()
                

